In [3]:
import soccerdata as sd
import pandas as pd
print("Setup works")

Setup works


In [4]:
import soccerdata as sd
import pandas as pd

In [5]:
import pandas as pd

url = "https://www.football-data.co.uk/mmz4281/2627/E0.csv"
fd = pd.read_csv(url)
print(fd.shape)
fd.head()

(100, 114)


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,BFEC<2.5,AHCh,B365CAHH,B365CAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,21/08/2026,20:00,Arsenal,Coventry,3,0,H,2,0,...,2.56,-2.00,2.05,1.75,2.05,1.83,2.02,1.74,2.14,1.86
1,E0,22/08/2026,12:30,Hull,Man United,2,0,H,2,0,...,2.56,1.50,1.88,1.98,1.88,1.98,1.82,1.92,1.92,2.06
2,E0,22/08/2026,15:00,Everton,Crystal Palace,2,0,H,1,0,...,1.92,-0.25,2.05,1.80,2.05,1.84,1.96,1.79,2.08,1.90
3,E0,22/08/2026,15:00,Ipswich,Sunderland,2,1,H,1,1,...,1.76,0.00,2.10,1.78,2.10,1.83,1.99,1.77,2.11,1.89
4,E0,22/08/2026,15:00,Nott'm Forest,Leeds,0,1,A,0,0,...,1.77,-0.25,2.10,1.70,2.10,1.75,2.07,1.70,2.19,1.83


In [6]:
print(len(fd))
print(sorted(fd["HomeTeam"].unique()))
print(fd[["Date", "HomeTeam", "AwayTeam"]].tail())

100
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Chelsea', 'Coventry', 'Crystal Palace', 'Everton', 'Fulham', 'Hull', 'Ipswich', 'Leeds', 'Liverpool', 'Man City', 'Man United', 'Newcastle', "Nott'm Forest", 'Sunderland', 'Tottenham']
          Date        HomeTeam     AwayTeam
95  08/11/2026        Brighton    Brentford
96  08/11/2026  Crystal Palace    Liverpool
97  08/11/2026         Ipswich  Bournemouth
98  08/11/2026      Sunderland      Chelsea
99  08/11/2026      Man United  Aston Villa


In [7]:
cols = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR",
        "HS", "AS", "HST", "AST", "HF", "AF", "HC", "AC", "HY", "AY", "HR", "AR"]

fd["Date"] = pd.to_datetime(fd["Date"], dayfirst=True)

united = fd[(fd["HomeTeam"] == "Man United") | (fd["AwayTeam"] == "Man United")]
carrick = united[united["Date"] < "2026-11-12"][cols].copy()
print(len(carrick))
carrick


10


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
1,2026-08-22,Hull,Man United,2,0,H,8,21,4,5,10,9,1,6,2,1,0,0
18,2026-08-30,Man United,Ipswich,5,2,H,33,9,11,5,7,12,8,4,1,3,0,0
28,2026-09-06,Everton,Man United,2,2,D,18,14,6,3,7,11,4,3,3,3,0,0
38,2026-09-13,Man United,Man City,0,1,A,16,6,3,2,17,10,6,7,1,2,0,1
49,2026-09-20,Fulham,Man United,1,1,D,12,29,6,6,9,5,3,13,3,0,0,0
55,2026-10-10,Man United,Tottenham,1,1,D,12,12,4,3,13,16,4,4,4,3,0,1
67,2026-10-18,Leeds,Man United,1,1,D,12,13,5,3,16,5,2,3,2,2,0,0
78,2026-10-25,Man United,Bournemouth,0,3,A,17,10,5,4,13,17,5,4,4,1,0,0
80,2026-10-31,Chelsea,Man United,2,0,H,19,12,10,5,8,20,8,7,1,6,0,1
99,2026-11-08,Man United,Aston Villa,4,4,D,26,12,9,6,18,9,8,1,1,1,0,0


In [8]:
carrick.to_csv("carrick_era_matches.csv", index=False)

In [9]:
import numpy as np

c = carrick.copy()
home = c["HomeTeam"] == "Man United"

c["Venue"] = np.where(home, "Home", "Away")
c["Opponent"] = np.where(home, c["AwayTeam"], c["HomeTeam"])
c["GF"] = np.where(home, c["FTHG"], c["FTAG"])
c["GA"] = np.where(home, c["FTAG"], c["FTHG"])
c["Shots"] = np.where(home, c["HS"], c["AS"])
c["ShotsAgainst"] = np.where(home, c["AS"], c["HS"])
c["SoT"] = np.where(home, c["HST"], c["AST"])
c["SoTAgainst"] = np.where(home, c["AST"], c["HST"])
c["Corners"] = np.where(home, c["HC"], c["AC"])
c["Fouls"] = np.where(home, c["HF"], c["AF"])

c["Points"] = np.select([c["GF"] > c["GA"], c["GF"] == c["GA"]], [3, 1], default=0)

c = c[["Date", "Opponent", "Venue", "GF", "GA", "Points", "Shots", "ShotsAgainst", "SoT", "SoTAgainst", "Corners", "Fouls"]]
c

,Date,Opponent,Venue,GF,GA,Points,Shots,ShotsAgainst,SoT,SoTAgainst,Corners,Fouls
1,2026-08-22,Hull,Away,0,2,0,21,8,5,4,6,9
18,2026-08-30,Ipswich,Home,5,2,3,33,9,11,5,8,7
28,2026-09-06,Everton,Away,2,2,1,14,18,3,6,3,11
38,2026-09-13,Man City,Home,0,1,0,16,6,3,2,6,17
49,2026-09-20,Fulham,Away,1,1,1,29,12,6,6,13,5
55,2026-10-10,Tottenham,Home,1,1,1,12,12,4,3,4,13
67,2026-10-18,Leeds,Away,1,1,1,13,12,3,5,3,5
78,2026-10-25,Bournemouth,Home,0,3,0,17,10,5,4,5,13
80,2026-10-31,Chelsea,Away,0,2,0,12,19,5,10,7,20
99,2026-11-08,Aston Villa,Home,4,4,1,26,12,9,6,8,18


In [10]:
summary = c[["GF", "GA", "Points", "Shots", "ShotsAgainst", "SoT", "SoTAgainst", "Corners", "Fouls"]].mean().round(2)
print(summary)
print("Total points:", c["Points"].sum())

GF               1.4
GA               1.9
Points           0.8
Shots           19.3
ShotsAgainst    11.8
SoT              5.4
SoTAgainst       5.1
Corners          6.3
Fouls           11.8
dtype: float64
Total points: 8


In [11]:
c.groupby("Venue")[["GF", "GA", "Points", "Shots", "ShotsAgainst", "SoT", "SoTAgainst"]].mean().round(2)

,GF,GA,Points,Shots,ShotsAgainst,SoT,SoTAgainst
Venue,,,,,,,
Away,0.8,1.6,0.6,17.8,13.8,4.4,6.2
Home,2.0,2.2,1.0,20.8,9.8,6.4,4.0


In [12]:
c.to_csv("carrick_era_clean.csv", index=False)